# Assignment: Learning Arithmetic with Sequence based models

In this assignment you will investigate whether sequence based models can learn a simple arithmetic algorithm only from examples. Specifically, you will compare a vanilla RNN and an LSTM on the task of adding integers with **a maximum of 2 digits**.  The point is not to build the best possible RNN or LSTM, but to investigate the properties of a simple setup.  

Here, addition will be learnt similar to how some language models are constructed.  Each training example (data point) consists of an arithmetic expression.  For example, consider 37+58 = 95.  In the left-to-right (LTR) representation, this is given as

```text
037+058=0095;
```

or, in the right-to-left (RTL) representation,

```text
730+850=5900;
```
where the digits in each number have been reversed. In both cases the underlying arithmetic problem is identical; the only difference is the order in which the digits are input to the network.

You will construct models that process the input one character at a time.The two architectures you will consider are

* **Vanilla RNN:** maintains a single hidden state that is updated at every timestep.
* **LSTM:** augments the hidden state with an internal memory state, candidate state and gating mechanisms that allow information to be retained or discarded over longer sequences.

You will compare how these architectures behave when learning the same arithmetic task under different input representations.   

## The standard carry algorithm

The familiar pencil-and-paper algorithm for addition proceeds from the **least significant** digit to the **most significant** digit. Let $a_t$ and $b_t$ denote the digits at position $t$ of the first and second number respectively, where $t=0$ corresponds to the units digit (i.e. rightmost digit). If $C_t$ denotes the carry into position $t$, then the addition algorithm is
$$
s_t = (a_t + b_t + C_t) \bmod 10,
$$
$$
C_{t+1} = \left\lfloor \frac{a_t + b_t + C_t}{10} \right\rfloor.
$$

For example,

```text
  278
+ 145
-----
```

is computed as follows:

* Units: (8 + 5 = 13). Write **3** and carry **1**.
* Tens: (7 + 4 + 1 = 12). Write **2** and carry **1**.
* Hundreds: (2 + 1 + 1 = 4). Write **4**.

i.e. $a_0 = 8, b_0 = 5, a_1 = 7, b_1 = 4, a_2 = 2, b_2 = 1, C_0 = 0$
* $t=0$: $8+5+0=13$, so $s_0=3$, carry $C_1=1$
* $t=1$: $7+4+1=12$, so $s_1=2$, carry $C_2=1$
* $t=2$: $2+1+1=4$, so $s_2=4$, carry $C_3=0$

So the sum is $s_2 s_1 s_0 = 423$.


An important observation is that the carry $C_t$ is the only information from previous digit positions that is required to compute the next digit. Once the carry is known, earlier digits are no longer needed.




##**Indicate your name and student number here**
**Name**:

**zID**:

## Submission instructions

This assignment contains model training and may take approximately **10--20 minutes** to run, depending on your hardware. You must submit an **executed Python notebook** in which all code-cell outputs, plots, tables, and reported metrics are visible. You are **not** required to submit the trained model files.

Before submission:

1. Complete all exercises and written responses.  Do **not** modify any existing code, only add where instructed.  Discussion questions should be completed in the text cell at the end of the assignment (see Further questions cell).
2. When ready, run all cells.
3. Check that every code cell runs without an error and that all requested plots, tables, and metrics are displayed beneath the relevant cells.
5. Save the notebook after execution, with the filename "MATHx881_zID_Lastname" us
6. Reopen the saved `.ipynb` file and confirm that the outputs are still visible before uploading it to Moodle.
7. Upload the executed `.ipynb` file to Moodle. Do not clear the notebook outputs before submission.

> **Important:** Your submission will be based only the displayed results without retraining, so it's important to ensure all results are visible.





## 1. Imports and configuration


In [ ]:
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

SEED = 12345
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


## 2. Create the vocabulary

Neural networks cannot operate directly on text or characters such as `'3'`, `'+'`, or `'='`. Instead, every character must first be converted into a numerical representation. This process is known as **tokenisation** or **encoding**.

In this assignment, each individual character is treated as a separate **token**. The set of all possible tokens is called the **vocabulary**, and each token is assigned a unique integer index. For example, the character `'0'` might be assigned the index `0`, `'1'` the index `1`, `'+'` the index `10`, and so on.

You will construct the vocabulary and create two lookup tables:

* a **character-to-index** mapping, which converts characters into integers that can be processed by the neural network; and
* an **index-to-character** mapping, which converts the network's numerical predictions back into readable text.

These mappings will be used throughout the remainder of the notebook whenever text is passed to the model or predictions are converted back into characters for inspection.

### **Exercise**

Complete the code cell below to do the following:

1. Create a list called `CHARS` containing, in order, the characters

   ```text
   0123456789+=;
   ```
2. Create a dictionary called `stoi` that maps each character to its integer index.
3. Create a dictionary called `itos` that maps each integer index back to its corresponding character.
4. Define `VOCAB_SIZE` to be the total number of characters in the vocabulary.
5. Write a function `encode(s)` that converts a string `s` into a list of integer indices.
6. Write a function `decode(ids)` that converts a sequence of integer indices back into a string.
7. Print the vocabulary, its size, and both lookup dictionaries to verify your implementation.



In [ ]:
##### INPUT YOUR CODE HERE:



## 3. Generate training data

Your next task is to create functions that randomly generate arithmetic examples in the format required by the models. Although the main experiments in this assignment consider two-digit addition, your implementation must work for an **arbitrary number of digits** specified by `num_digits`.

Numbers containing fewer than `num_digits` digits must be padded with zeros on the left. For example, when `num_digits=3`, the number (37) should be represented as `037`.

Each dataset element should contain a complete character string of the form (in the case of 3 digit number addition)

```text
aaa+bbb=cccc;
```

where the answer is represented using `num_digits + 1` digits because the sum of two `num_digits`-digit numbers may require one additional digit.

You will consider two representations.

### Left-to-right representation

The digits appear in their usual order:

```text
037+058=0095;
```

### Right-to-left representation

The digits of both operands and the answer are reversed:

```text
730+850=5900;
```

The symbols `+`, `=`, and `;` remain in the same relative order.

###**Exercise**

Complete the code cell below by defining the following two functions.

#### 1. Write a function in the following form

```python
format_example(a, b, num_digits=3, direction="ltr")
```

This function must

1. pad each number to exactly `num_digits` digits;
2. pad the answer to exactly `num_digits + 1` digits;
3. reverse the digits of the numbers and answer when `direction="rtl"`;
4. construct:

   * the input expression ending in `=`,
   * the target answer string,
   * the complete expression ending in `;`;
5. return these three strings.

The function should accept only `"ltr"` and `"rtl"` as valid directions.

#### 2. Write another function in the form

```python
generate_examples(n_examples, num_digits=3, direction="ltr", seed=0)
```

This function must

1. randomly sample `n_examples` pairs of non-negative integers representable using at most `num_digits` digits;
2. format each pair using `format_example`;
3. record the numbers, their sum, the formatted strings, the number of digits, and the representation direction;
4. return the examples as a `pandas.DataFrame`.

Finally, generate and print five examples of adding two three-digit numbers for each of the left-to-right and right-to-left representations.



In [ ]:
##### INPUT YOUR CODE HERE:




## 4. Carry metadata

The below cell collates information about the carry in the pen and pencil addition algorithm.

Recall that for two integers written in the usual decimal notation, the carry algorithm processes digits from the units position to the most significant position. Let $a_t$ and $b_t$ denote the digits at position $t$, and let $C_t\in\{0,1\}$ be the carry entering that position, with $C_0=0$. It computes

$$
s_t=(a_t+b_t+C_t)\bmod 10,
\qquad
C_{t+1}=\left\lfloor\frac{a_t+b_t+C_t}{10}\right\rfloor.
$$

The list `carry_out` contains the values

$$
[C_1,C_2,\ldots,C_D],
$$

where $D$ is the number of input digits. Thus, `carry_out[t]` is the carry produced after adding the digits at position $t$. The quantity `n_carries` is the total number of positions that generate a carry, while `max_carry_chain` is the longest run of consecutive carry outputs equal to one.


In [ ]:
def carry_sequence(a, b, num_digits=3):
    carry = 0
    carry_in = []
    carry_out = []

    for t in range(num_digits):
        da = (a // (10 ** t)) % 10
        db = (b // (10 ** t)) % 10

        carry_in.append(carry)
        total = da + db + carry
        carry = total // 10
        carry_out.append(carry)

    return carry_in, carry_out


def max_carry_chain_length(a, b, num_digits=3):
    _, carry_out = carry_sequence(a, b, num_digits)
    best = 0
    current = 0
    for c in carry_out:
        if c == 1:
            current += 1
            best = max(best, current)
        else:
            current = 0
    return best


def add_carry_metadata(df):
    df = df.copy()
    carry_ins, carry_outs, n_carries, max_chains = [], [], [], []

    for row in df.itertuples(index=False):
        ci, co = carry_sequence(row.a, row.b, row.num_digits)
        carry_ins.append(ci)
        carry_outs.append(co)
        n_carries.append(sum(co))
        max_chains.append(max_carry_chain_length(row.a, row.b, row.num_digits))

    df["carry_in"] = carry_ins
    df["carry_out"] = carry_outs
    df["n_carries"] = n_carries
    df["max_carry_chain"] = max_chains
    return df


demo = add_carry_metadata(generate_examples(8, num_digits=3, direction="ltr", seed=7))
demo[["a", "b", "sum", "text", "carry_in", "carry_out", "n_carries", "max_carry_chain"]]


## 5. Dataset

The following cell makes use of the above functions and generates minibatches for training.

In [ ]:
class AdditionLanguageModelDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.encoded = [torch.tensor(encode(s), dtype=torch.long) for s in self.df["text"]]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ids = self.encoded[idx]

        x = ids[:-1]
        y = ids[1:]

        # In y, the answer digits start just after the '=' in the original string.
        # x has length len(text)-1, y[t] is text[t+1].
        # For string aaa+bbb=cccc;, the first answer digit is y[2*num_digits + 1].
        answer_start_in_y = 2 * int(row["num_digits"]) + 1
        answer_end_in_y = answer_start_in_y + int(row["num_digits"]) + 1

        meta = {
            "a": int(row["a"]),
            "b": int(row["b"]),
            "sum": int(row["sum"]),
            "input": row["input"],
            "target": row["target"],
            "text": row["text"],
            "direction": row["direction"],
            "num_digits": int(row["num_digits"]),
            "answer_start_in_y": int(answer_start_in_y),
            "answer_end_in_y": int(answer_end_in_y),
            "n_carries": int(row.get("n_carries", -1)),
            "max_carry_chain": int(row.get("max_carry_chain", -1)),
            "carry_in": row.get("carry_in", None),
            "carry_out": row.get("carry_out", None),
        }

        return x, y, meta


def collate_batch(batch):
    xs, ys, metas = zip(*batch)
    return torch.stack(xs), torch.stack(ys), list(metas)

## 6. Model setup

The below code defines the model structure (RNN and LSTM) that you will be training. Its structure is given by

$$
x_t \rightarrow \text{embedding} \rightarrow \text{RNN/LSTM} \rightarrow \text{linear logits for }x_{t+1}.
$$

In [ ]:
class CharLanguageModel(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, hidden_dim=128, cell_type="rnn"):
        super().__init__()
        assert cell_type in {"rnn", "lstm"}
        self.cell_type = cell_type

        self.embedding = nn.Embedding(vocab_size, emb_dim)

        if cell_type == "rnn":
            self.recurrent = nn.RNN(
                input_size=emb_dim,
                hidden_size=hidden_dim,
                batch_first=True,
                nonlinearity="tanh",
            )
        else:
            self.recurrent = nn.LSTM(
                input_size=emb_dim,
                hidden_size=hidden_dim,
                batch_first=True,
            )

        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, return_hidden=False):
        emb = self.embedding(x)

        if self.cell_type == "rnn":
            hidden_seq, h_final = self.recurrent(emb)
        else:
            hidden_seq, (h_final, c_final) = self.recurrent(emb)

        logits = self.output(hidden_seq)

        if return_hidden:
            return logits, hidden_seq

        return logits

## 7. Training and evaluation

The code below defines the functions used for training (including validation). The accuracy metrics are:

### 1. Answer-digit accuracy

Suppose the evaluation dataset contains $M$ arithmetic examples. Let each answer contain $K$ characters. If $\hat{y}_{i,k}$ and $y_{i,k}$ are the predicted and true digits in the $k$th position of the $i$th digit respectively then  

$$\frac{1}{MK} \sum_{i=1}^M \sum_{k=1}^K {\mathbf{1}}_{\hat{y}_{i,k} = y_{i,k}} $$
is the answer digit accuracy.
This measures the proportion of individual answer digits predicted correctly. It gives partial credit when only some digits of the final answer are correct.

### 2. Exact-answer accuracy

For $M$ arithmetic examples, let $\hat{y}_i$ and $y_i$ denote the complete predicted and true answer strings respectively. Then

$$\frac{1}{M} \sum_{i=1}^M  {\mathbf{1}}_{\hat{y}_{i} = y_{i}} $$

is the exact answer accuracy.
This is stricter: an example is counted as correct only when every answer digit is correct.


In [ ]:
def lm_loss(logits, y):
    return nn.functional.cross_entropy(logits.reshape(-1, VOCAB_SIZE), y.reshape(-1))


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    total_tokens = 0

    for x, y, _ in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(x)
        loss = lm_loss(logits, y)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * y.numel()
        total_tokens += y.numel()

    return total_loss / total_tokens


@torch.no_grad()
def evaluate_answer_region(model, loader):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    answer_digit_correct = 0
    answer_digit_total = 0
    exact_answer_correct = 0
    total_sequences = 0

    rows = []

    for x, y, metas in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x)
        loss = lm_loss(logits, y)
        pred = logits.argmax(dim=-1)

        total_loss += loss.item() * y.numel()
        total_tokens += y.numel()

        for i, meta in enumerate(metas):
            start = meta["answer_start_in_y"]
            end = meta["answer_end_in_y"]

            pred_answer_ids = pred[i, start:end].cpu().tolist()
            true_answer_ids = y[i, start:end].cpu().tolist()

            pred_answer = decode(pred_answer_ids)
            true_answer = decode(true_answer_ids)

            correct_digits = sum(int(p == t) for p, t in zip(pred_answer_ids, true_answer_ids))
            answer_digit_correct += correct_digits
            answer_digit_total += len(true_answer_ids)

            is_correct = pred_answer == true_answer
            exact_answer_correct += int(is_correct)
            total_sequences += 1

            rows.append({
                **meta,
                "predicted": pred_answer,
                "correct": bool(is_correct),
            })

    return {
        "loss": total_loss / total_tokens,
        "answer_digit_accuracy": answer_digit_correct / answer_digit_total,
        "exact_answer_accuracy": exact_answer_correct / total_sequences,
        "predictions": pd.DataFrame(rows),
    }


def run_training(model, train_loader, val_loader, epochs=45, lr=3e-3):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = []

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer)

        # Re-evaluate the updated model without gradient tracking so that
        # training and validation accuracies are directly comparable.
        train_metrics = evaluate_answer_region(model, train_loader)
        val_metrics = evaluate_answer_region(model, val_loader)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_eval_loss": train_metrics["loss"],
            "train_answer_digit_accuracy": train_metrics["answer_digit_accuracy"],
            "train_exact_answer_accuracy": train_metrics["exact_answer_accuracy"],
            "val_loss": val_metrics["loss"],
            "val_answer_digit_accuracy": val_metrics["answer_digit_accuracy"],
            "val_exact_answer_accuracy": val_metrics["exact_answer_accuracy"],
        }
        history.append(row)

        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            print(
                f"Epoch {epoch:3d} | train loss {train_loss:.4f} | "
                f"val answer digit acc {row['val_answer_digit_accuracy']:.3f} | "
                f"val exact answer acc {row['val_exact_answer_accuracy']:.3f}"
            )

    return model, pd.DataFrame(history)

## 8. Configuration

This section specifies the main hyperparameters and dataset sizes. You will focus on two-digit addition.

### **Exercise**

Specify appropriate values for each of the hyperparameters below, in the next code cell.

- `num_digits_train`: number of digits in each of the two integers used for training. A value of 2 means that each number lies between 00 and 99.
- `n_train`: number of randomly generated training examples used to update the model parameters.
- `n_val`: number of validation examples used during training to monitor generalisation (out of sample performance) and compare model configurations. These examples are not used by the optimiser.
- `n_test`: number of held-out test examples used only for the final performance assessment.
- `batch_size`: number of sequences processed in one optimisation step.
- `emb_dim`: dimension of the learned vector embedding assigned to each input character.
- `hidden_dim`: number of components in the RNN or LSTM hidden state.
- `epochs`: number of complete passes through the training dataset.
- `lr`: Adam learning rate, which controls the scale of each parameter update.

The statement `cfg = Config()` creates one configuration object containing these values.


In [ ]:
#### INPUT YOUR CHOSEN HYPERPARAMETERS HERE:

@dataclass
class Config:
    num_digits_train: int =
    n_train: int =
    n_val: int =
    n_test: int =
    batch_size: int =
    emb_dim: int =
    hidden_dim: int =
    epochs: int =
    lr: float = 3e-3

cfg = Config()
cfg

## 9. Build datasets

The below code builds the LTR and RTL representation datasets based on previously defined functions.

In [ ]:
def make_loaders(direction, num_digits, seed_offset=0):
    train_df = generate_examples(cfg.n_train, num_digits=num_digits, direction=direction, seed=SEED + seed_offset)
    val_df = generate_examples(cfg.n_val, num_digits=num_digits, direction=direction, seed=SEED + 1000 + seed_offset)
    test_df = generate_examples(cfg.n_test, num_digits=num_digits, direction=direction, seed=SEED + 2000 + seed_offset)

    train_df = add_carry_metadata(train_df)
    val_df = add_carry_metadata(val_df)
    test_df = add_carry_metadata(test_df)

    train_ds = AdditionLanguageModelDataset(train_df)
    val_ds = AdditionLanguageModelDataset(val_df)
    test_ds = AdditionLanguageModelDataset(test_df)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_batch)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_batch)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_batch)

    return train_df, val_df, test_df, train_loader, val_loader, test_loader


datasets = {}
for direction in ["ltr", "rtl"]:
    datasets[direction] = make_loaders(direction, cfg.num_digits_train, seed_offset=0 if direction == "ltr" else 777)
    print("\n", direction)
    print(datasets[direction][0][["text", "input", "target", "a", "b", "sum"]].head())

## 10. Train four models

Train four models:

1. RNN on RTL representation data
2. LSTM on RTL representation data
3. RNN on LTR representation data
4. LSTM on LTR representation data

The validation set is evaluated repeatedly during training and is used to monitor learning and diagnose overfitting. The test set is held aside and is evaluated only after training has finished.


In [ ]:
def build_model(cell_type):
    return CharLanguageModel(
        vocab_size=VOCAB_SIZE,
        emb_dim=cfg.emb_dim,
        hidden_dim=cfg.hidden_dim,
        cell_type=cell_type,
    )

models = {}
histories = {}
test_results = {}

for direction in ["ltr", "rtl"]:
    train_df, val_df, test_df, train_loader, val_loader, test_loader = datasets[direction]

    for cell_type in ["rnn", "lstm"]:
        key = f"{cell_type}_{direction}"

        print("\n" + "=" * 80)
        print("Training", key)
        print("=" * 80)

        torch.manual_seed(SEED)
        model = build_model(cell_type)

        model, hist = run_training(
            model,
            train_loader,
            val_loader,
            epochs=cfg.epochs,
            lr=cfg.lr,
        )

        test_metrics = evaluate_answer_region(model, test_loader)

        print("\nTest metrics:", {
            "loss": test_metrics["loss"],
            "answer_digit_accuracy": test_metrics["answer_digit_accuracy"],
            "exact_answer_accuracy": test_metrics["exact_answer_accuracy"],
        })

        models[key] = model
        histories[key] = hist
        test_results[key] = test_metrics

## 11. Compare results



###**Exercise**

Using the data frames stored in the dictionary `histories`, construct a $ 2\times2$ figure showing how the four trained models perform over the training epochs.

Your figure must contain the following panels:

1. training exact-answer accuracy;
2. validation exact-answer accuracy;
3. training answer-digit accuracy;
4. validation answer-digit accuracy.

Next, use the dictionary `test_results` to construct a summary table containing one row for each trained model. The table must report:

* model type;
* representation direction;
* test loss;
* test answer-digit accuracy;
* test exact-answer accuracy.

Sort the rows first by representation direction and then by model type.



In [ ]:
##### INPUT YOUR CODE HERE:


## 12. Inspect predictions for each model

This prints the task, true answer, predicted answer, and correctness for each of the four models.


In [ ]:
for key, metrics in test_results.items():
    print("\n" + "=" * 80)
    print(key)
    print("=" * 80)

    display(
        metrics["predictions"][
            ["input", "target", "predicted", "correct", "n_carries", "max_carry_chain"]
        ].head(20)
    )

## 13. Compare all four models on aligned examples

Notice that in the test set used earlier, the LTR and RTL test sets do not necessarily contain the same underlying arithmetic problems. Here you will create a small shared set of addition problems so that the predictions of all four models can be compared example by example. This is a diagnostic stress test and does not replace the main test-set evaluation.

###**Exercise**

Create 20 arithmetic problems that are shared across all four trained models.

Construct both the LTR and RTL representations of each underlying pair $a+b$. Evaluate the four models, i.e.

1. RNN trained on LTR data;
2. LSTM trained on LTR data;
3. RNN trained on RTL data;
4. LSTM trained on RTL data.

Display an example-by-example table containing the two numbers, true sum, true answer in each representation, and the four model predictions.

Then construct a summary table reporting answer-digit accuracy and exact-answer accuracy for each model.



In [ ]:
##### INPUT YOUR CODE HERE:




## 17. Further questions:

Provide at most one paragraph answers to each of the following questions.  Write your responses directly below each question.

1. What is the loss function that has been adopted in this assignment? Describe it using equations. Does this correspond to a "many to many" sequence learning problem or a "many to one" sequence problem?



2. What do you observe about performance in the RTL vs LTR representation based models? Explain your observations by referring to the underlying learning problem.



3. Explain the behaviour you observe when comparing performance of the RNN and LSTM.  Consider both the LTR and RTL representations.


